# Characterizing the customer
Our objective is to characterize groups of customers based on their
buys. We will check which data from Olist data set can be used for this
reason.

In [1]:
## python built-in modules
from pathlib import Path

## third party modules
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

## loads specific package modules, which contain project settings
from client_segmentation.config import PROCESSED_DATA_DIR
from client_segmentation.config import RAW_DATA_DIR
from client_segmentation.config import INTERIM_DATA_DIR

2025-04-24 14:36:37.451 | INFO     | client_segmentation.config:<module>:11 - PROJ_ROOT path is: /home/gasobral/Meus Arquivos/data-science/client_segmentation


In [2]:
## below there is a mapping of dtypes for every column of each csv file
## data_mapping is a dictionary which its key = csv file and
## value = dict (dtype mapping, where key = column and value = dtype)
product_category_name_translation_map = {
    'product_category_name' : 'string',
    'product_category_name_english' : 'string'
}

olist_sellers_dataset_map = {
    'seller_id': 'string',
    'seller_zip_code_prefix': 'int64',
    'seller_city': 'string',
    'seller_state': 'string'
}

olist_geolocation_dataset_map = {
    'geolocation_zip_code_prefix': 'int64',
    'geolocation_lat': 'float64',
    'geolocation_lng': 'float64',
    'geolocation_city': 'string',
    'geolocation_state': 'string'
}

olist_products_dataset_map = {
    'product_id': 'string',
    'product_category_name': 'string',
    'product_name_lenght': 'float64',
    'product_description_lenght': 'float64',
    'product_photos_qty':  'float64',
    'product_weight_g': 'float64',
    'product_length_cm': 'float64',
    'product_height_cm': 'float64',
    'product_width_cm': 'float64'
}

olist_order_items_dataset_map = {
    'order_id': 'string',
    'order_item_id': 'int64',
    'product_id': 'string',
    'seller_id': 'string',
    'shipping_limit_date': 'object',
    'price': 'float64',
    'freight_value': 'float64'
}

olist_order_payments_dataset_map = {
    'order_id': 'string',
    'payment_sequential': 'int64',
    'payment_type': 'string',
    'payment_installments': 'int64',
    'payment_value': 'float64',
}

olist_orders_dataset_map = {
    'order_id': 'string',
    'customer_id': 'string',
    'order_status': 'string',
    'order_purchase_timestamp': 'object',
    'order_approved_at': 'object',
    'order_delivered_carrier_date': 'object',
    'order_delivered_customer_date': 'object',
    'order_estimated_delivery_date': 'object'
}

olist_order_reviews_dataset_map = {
    'review_id': 'string',
    'order_id': 'string',
    'review_score': 'int64',
    'review_comment_title': 'string',
    'review_comment_message': 'string',
    'review_creation_date': 'object',
    'review_answer_timestamp': 'object'
}

olist_customers_dataset_map = {
     'customer_id': 'string',
     'customer_unique_id': 'string',
     'customer_zip_code_prefix': 'int64',
     'customer_city': 'string',
     'customer_state': 'string'
}

data_mapping = {
    'product_category_name_translation': product_category_name_translation_map,
    'olist_sellers_dataset': olist_sellers_dataset_map,
    'olist_geolocation_dataset': olist_geolocation_dataset_map,
    'olist_products_dataset': olist_products_dataset_map,
    'olist_order_items_dataset': olist_order_items_dataset_map,
    'olist_order_payments_dataset': olist_order_payments_dataset_map,
    'olist_orders_dataset': olist_orders_dataset_map,
    'olist_order_reviews_dataset': olist_order_reviews_dataset_map,
    'olist_customers_dataset': olist_customers_dataset_map
}

In [3]:
def data_aquisition(file_path: Path,
                    data_mapping: dict) -> pd.DataFrame:
    """
    Given a file path, reads a dataset and return a data frame.
    
    Arguments
    ---------
    file_path: a file path for a csv file.
    
    data_mapping: a dictionary with a mapping, which tells type of data
                  for each column of csv file.

    Return
    ------
    A data frame with the data loaded from csv file.
    """

    ## create file without file extention (suffix) in order to
    ## obtain the date mapping for dataset columns 
    name_suffixless = file_path.name.split('.')[0]
    dataset = pd.read_csv(file_path,
                          dtype=data_mapping[name_suffixless],
                          date_format="%Y-%m-%d %H:%M:%S")

    ## creates a list only with the columns which contain data on
    ## their name
    date_columns = [col for col in dataset.columns
                    if "date" in col or
                    "time" in col or
                    "order_approved_at" in col]

    if len(date_columns) != 0:
        for col in date_columns:
            dataset[col] = pd.to_datetime(dataset[col],
                                          format="%Y-%m-%d %H:%M:%S")
    else:
        print("Dataset has no columns which contain date!"
              " No parsing required.")

    return dataset

## Analyzing customer data

In [4]:
customer = data_aquisition(INTERIM_DATA_DIR / 'olist_customers_dataset.csv',
                           data_mapping)

Dataset has no columns which contain date! No parsing required.


In [5]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: int64(1), string(4)
memory usage: 3.8 MB


In [6]:
customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


For customer specific data, we only use its location data (city and state),
since we do not have their names or any personal information. Now let's check
which kind of goods a customer buys/orders.

Regarding data schema shown in
[Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce),
product specific data is located at olist_product_dataset. Moreover, in
order to know which products a customer order, we must combine the datasets
olist_order_customer_dataset, olist_orders_dataset,
olist_order_itens_dataset and olist_produts_dataset.

In [7]:
order = data_aquisition(INTERIM_DATA_DIR / 'olist_orders_dataset.csv',
                        data_mapping)

order_items = data_aquisition(INTERIM_DATA_DIR / 'olist_order_items_dataset.csv',
                              data_mapping)

products = data_aquisition(INTERIM_DATA_DIR / 'olist_products_dataset.csv',
                           data_mapping)

Dataset has no columns which contain date! No parsing required.


In [8]:
customer_order = pd.merge(customer,
                          order,
                          how='inner',
                          on='customer_id')

order_products = pd.merge(order_items,
                          products,
                          how='inner',
                          on='product_id')

customer_products = pd.merge(customer_order,
                             order_products,
                             how='inner',
                             on='order_id')

In [9]:
customer_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   customer_id                    112650 non-null  string        
 1   customer_unique_id             112650 non-null  string        
 2   customer_zip_code_prefix       112650 non-null  int64         
 3   customer_city                  112650 non-null  string        
 4   customer_state                 112650 non-null  string        
 5   order_id                       112650 non-null  string        
 6   order_status                   112650 non-null  string        
 7   order_purchase_timestamp       112650 non-null  datetime64[ns]
 8   order_approved_at              112635 non-null  datetime64[ns]
 9   order_delivered_carrier_date   111456 non-null  datetime64[ns]
 10  order_delivered_customer_date  110196 non-null  datetime64[ns]
 11  

In [10]:
customer_products.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,124.99,21.88,moveis_escritorio,41.0,1141.0,1.0,8683.0,54.0,64.0,31.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,289.00,46.48,utilidades_domesticas,43.0,1002.0,3.0,10150.0,89.0,15.0,40.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,139.94,17.79,moveis_escritorio,55.0,955.0,1.0,8267.0,52.0,52.0,17.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,149.94,23.36,moveis_escritorio,48.0,1066.0,1.0,12160.0,56.0,51.0,28.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,230.00,22.25,casa_conforto,61.0,407.0,1.0,5200.0,45.0,15.0,35.0


In [11]:
## 1) remoção de colunas
## campos que contêm ids
## retirar o zip code e usar apenas a cidade e estado (não usar informações redudantes)
## campos com delivery date devem ser removidos?
## fazer um contador de número de compras por tipo de ordem
## dos produtos, usar apenas o nome da categoria

## criar um registro (uma linha) para cada perfil de conta, data set final

In [12]:
## montar um data frame em que cada linha contém:
## [1]                                 [2]                         [3] 
## customer_unique_id, cidade, estado, total gasto (item + frete), número de itens comprados por categoria
## há aproximadamente 70 categorias (usa pca)
## [1] dados vem da olist_order_customer_dataset
## [2] valor agregado (olist_order_customer_dataset, olist_orders_dataset, olist_order_itens_dataset)

In [13]:
## prototipando o [1]
customer_itens = pd.merge(customer_order,
                          order_items,
                          how='inner',
                          on='order_id')

In [14]:
customer_itens.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05,1,a9516a079e37a9c9c36b9b78b10169e8,7c67e1448b00f6e969d365cea6b010ab,2017-05-22 15:22:12,124.99,21.88
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-01-18 20:58:32,289.00,46.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13,1,bd07b66896d6f1494f5b86251848ced7,7c67e1448b00f6e969d365cea6b010ab,2018-06-05 16:19:10,139.94,17.79
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10,1,a5647c44af977b148e0a3a4751a09e2e,7c67e1448b00f6e969d365cea6b010ab,2018-03-27 16:31:16,149.94,23.36
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15,1,9391a573abe00141c56e38d84d7d5b3b,4a3ca9315b744ce9f8e9374361493884,2018-07-31 10:10:09,230.00,22.25


In [15]:
customer_itens.groupby(by=['customer_unique_id'])['price'].sum()

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2     129.90
0000b849f77a49e4a4ce2b2a4ca5be3f      18.90
0000f46a3911fa3c0805444483337064      69.00
0000f6ccb0745a6a4b88665a16c9f078      25.99
0004aac84e0df4da2b147fca70cf8255     180.00
                                     ...   
fffcf5a5ff07b0908bd4e2dbc735a684    1570.00
fffea47cd6d3cc0a88bd621562a9d061      64.89
ffff371b4d645b6ecea244b27531430a      89.90
ffff5962728ec6157033ef9805bacc48     115.00
ffffd2657e2aad2907e67c3e9daecbeb      56.99
Name: price, Length: 95420, dtype: float64

In [16]:
## todos os clientes possuem uma ordem, portanto posso fazer o inner join
(~customer['customer_id'].isin(order['customer_id'])).sum()

0

In [17]:
customer.shape[0]

99441

In [18]:
customer_order.shape[0]

99441

In [19]:
customer_itens.shape[0]

112650

In [20]:
## o join com order_itens introduz alguns customer_unique_id's
## possivelemente valores duplicados
customer_itens['customer_unique_id'].nunique()

95420

In [21]:
customer_itens['customer_unique_id'].duplicated().sum()

17230

In [22]:
## analisando os registros duplicados
query = customer_itens['customer_unique_id'].duplicated()
customer_itens[query]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
22,690172ab319622688d3b4df42f676898,a96d5cfa0d3181817e2b946f921ea021,74914,aparecida de goiania,GO,aaff8afa47c8426e414a6d908a97713c,delivered,2017-10-15 11:08:48,2017-10-15 11:25:49,2017-10-16 21:36:29,2017-10-25 22:30:58,2017-11-06,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2017-10-19 11:25:49,59.90,17.67
23,690172ab319622688d3b4df42f676898,a96d5cfa0d3181817e2b946f921ea021,74914,aparecida de goiania,GO,aaff8afa47c8426e414a6d908a97713c,delivered,2017-10-15 11:08:48,2017-10-15 11:25:49,2017-10-16 21:36:29,2017-10-25 22:30:58,2017-11-06,3,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2017-10-19 11:25:49,59.90,17.67
36,b2bed119388167a954382cca36c4777f,e079b18794454de9d2be5c12b4392294,27525,resende,RJ,77b062be7c5bd21712905feb8e1cfeed,delivered,2017-06-14 18:31:54,2017-06-15 18:43:04,2017-06-22 08:11:38,2017-07-07 20:32:47,2017-07-07,2,17606c7d7254ed1f0351fd48a28be932,1900267e848ceeba8fa32d80c1a5f5a8,2017-06-21 18:43:04,44.99,16.14
49,19cecb194f54e614b70d971306a9931b,d251c190ca75786e9ab937982d60d1d4,30320,belo horizonte,MG,14282bc70be9bdda515182fb1ce62af4,delivered,2018-04-18 14:18:09,2018-04-19 02:52:02,2018-04-20 00:47:44,2018-04-26 16:26:38,2018-05-11,2,b05e00841a6dad404ef34ae67807879a,aac29b1b99776be73c3049939652091d,2018-04-25 02:51:32,11.99,13.47
61,aa9f03ecd3728c9bd12e6d962c66c7cb,b03e9d9818ee170e9d6b983803c7d406,75388,trindade,GO,07429f7601b56a66d1854c79a2f5c9e5,delivered,2017-07-11 09:36:46,2017-07-11 09:50:30,2017-07-11 19:42:28,2017-07-21 20:06:22,2017-08-04,2,a04a2402a7774ba53d10ed2ce29715e3,17ca9b9e9b9ef8fdb529001b49ebb50f,2017-07-17 09:50:30,99.97,16.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112622,b7c889215de76857c7967c1011125d2d,522e244a96d13876c5bac4985a8d5075,82410,curitiba,PR,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,3,e8316a4667e5870c85e906b1f062bde1,7c67e1448b00f6e969d365cea6b010ab,2018-02-14 15:30:30,79.99,30.40
112623,b7c889215de76857c7967c1011125d2d,522e244a96d13876c5bac4985a8d5075,82410,curitiba,PR,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,4,e8316a4667e5870c85e906b1f062bde1,7c67e1448b00f6e969d365cea6b010ab,2018-02-14 15:30:30,79.99,30.40
112624,b7c889215de76857c7967c1011125d2d,522e244a96d13876c5bac4985a8d5075,82410,curitiba,PR,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,5,e8316a4667e5870c85e906b1f062bde1,7c67e1448b00f6e969d365cea6b010ab,2018-02-14 15:30:30,79.99,30.40
112625,b7c889215de76857c7967c1011125d2d,522e244a96d13876c5bac4985a8d5075,82410,curitiba,PR,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,6,e8316a4667e5870c85e906b1f062bde1,7c67e1448b00f6e969d365cea6b010ab,2018-02-14 15:30:30,79.99,30.40


In [23]:
## o order_id é o motivo dos valores duplicados (pois uma compra contém mais de um item)
## para fazer essa conta, dropar os valores duplicados do order_id antes de fazer o merge
query = customer_itens['customer_unique_id'] == 'a96d5cfa0d3181817e2b946f921ea021'
customer_itens[query]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
21,690172ab319622688d3b4df42f676898,a96d5cfa0d3181817e2b946f921ea021,74914,aparecida de goiania,GO,aaff8afa47c8426e414a6d908a97713c,delivered,2017-10-15 11:08:48,2017-10-15 11:25:49,2017-10-16 21:36:29,2017-10-25 22:30:58,2017-11-06,1,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2017-10-19 11:25:49,59.9,17.67
22,690172ab319622688d3b4df42f676898,a96d5cfa0d3181817e2b946f921ea021,74914,aparecida de goiania,GO,aaff8afa47c8426e414a6d908a97713c,delivered,2017-10-15 11:08:48,2017-10-15 11:25:49,2017-10-16 21:36:29,2017-10-25 22:30:58,2017-11-06,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2017-10-19 11:25:49,59.9,17.67
23,690172ab319622688d3b4df42f676898,a96d5cfa0d3181817e2b946f921ea021,74914,aparecida de goiania,GO,aaff8afa47c8426e414a6d908a97713c,delivered,2017-10-15 11:08:48,2017-10-15 11:25:49,2017-10-16 21:36:29,2017-10-25 22:30:58,2017-11-06,3,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2017-10-19 11:25:49,59.9,17.67


In [24]:
order_items_uniq = order_items.drop_duplicates(subset=['order_id'],
                                               keep='first')

In [25]:
order_items_uniq['order_id'].duplicated().sum()

0

In [26]:
customer_itens_uniq = pd.merge(customer_order,
                               order_items_uniq,
                               how='inner',
                               on='order_id')

In [27]:
## fazer um join usando os dados abaixo
customer_itens_uniq.groupby(by=['customer_unique_id'])['price'].sum().reset_index()

,customer_unique_id,price
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90
2,0000f46a3911fa3c0805444483337064,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,25.99
4,0004aac84e0df4da2b147fca70cf8255,180.00
...,...,...
95415,fffcf5a5ff07b0908bd4e2dbc735a684,890.00
95416,fffea47cd6d3cc0a88bd621562a9d061,64.89
95417,ffff371b4d645b6ecea244b27531430a,89.90
95418,ffff5962728ec6157033ef9805bacc48,115.00
